# 📊 LinkedIn Jobs Market Analysis

MongoDB 및 덤프 파일(`data/silver_jobs.gz`)을 활용해 채용 시장 동향을 시각화하고 분석합니다.

### 1. 환경 설정 및 라이브러리 로드
이 주피터 노트북 환경에 `pymongo`, `pandas`, `matplotlib`, `seaborn` 라이브러리가 필요합니다.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient

# 시각화 스타일 설정
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.family'] = 'NanumGothic' or 'sans-serif'

### 2. 데이터 로드 (로컬 덤프 파일 확인 및 DB 연결)
로컬에 백업을 위해 추출한 `data/silver_jobs.gz`를 활용할 수 있습니다. `mongorestore`를 통해 DB로 다시 집어넣은 뒤 쿼리하거나, 실시간 MongoDB 커넥션을 직접 이용합니다.

In [ ]:
dump_path = "../data/silver_jobs.gz"
if os.path.exists(dump_path):
    print(f"✅ 로컬 덤프 파일 확인됨: {dump_path}")
    print("💡 덤프 복원 방법: 'mongorestore --gzip --archive=data/silver_jobs.gz' 실행 후 아래 셀을 구동하십시오.")

# MongoDB 연결 및 로드
try:
    client = MongoClient("mongodb://mongodb:27017")
    db = client.linkedin
    jobs_coll = db['silver.jobs']
    df = pd.DataFrame(list(jobs_coll.find()))
    if not df.empty and '_id' in df.columns:
        df = df.drop(columns=['_id'])
    print(f"📊 분석 대상 데이터 수: {len(df)}개")
except Exception as e:
    print("❌ MongoDB 접속에 실패했습니다.", e)
    df = pd.DataFrame()

df.head()

### 3. 주요 국가/지역별 채용 공고 분포 (Top 10)

In [ ]:
if not df.empty:
    top_geos = df['geo'].value_counts().head(10)
    sns.barplot(x=top_geos.values, y=top_geos.index, palette="viridis")
    plt.title("Top 10 채용 공고 국가/지역 분포", fontsize=15)
    plt.xlabel("공고 수")
    plt.ylabel("국가/지역")
    plt.show()
else:
    print("데이터가 없습니다.")

### 4. 근무 형태(Work Style) 분석

In [ ]:
if not df.empty and 'workStyle' in df.columns:
    workstyle_counts = df['workStyle'].value_counts()
    plt.pie(workstyle_counts, labels=workstyle_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
    plt.title("근무 형태(Work Style) 비율", fontsize=15)
    plt.axis('equal')
    plt.show()

### 5. 주요 채용 기업 (Top 10)

In [ ]:
if not df.empty:
    top_companies = df['companyName'].value_counts().head(10)
    sns.barplot(x=top_companies.values, y=top_companies.index, palette="magma")
    plt.title("가장 활발하게 채용 중인 기업 Top 10", fontsize=15)
    plt.xlabel("등록된 공고 수")
    plt.ylabel("기업명")
    plt.show()